# Keyword categories

Profile keyword categories across the full-endpoint publication corpus.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_02_keyword_categories")

rows = []
for category, pattern in CATEGORY_PATTERNS.items():
    hits = df["analysis_text"].apply(lambda text: contains_pattern(text, pattern))
    rows.append(
        {
            "category": category,
            "n_publications": len(df),
            "n_with_category": int(hits.sum()),
            "percent_with_category": float(hits.mean() * 100),
        }
    )

category_summary = pd.DataFrame(rows).sort_values("percent_with_category")
category_summary.to_csv(table_dir / "keyword_category_summary.csv", index=False)

figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(category_summary["category"], category_summary["percent_with_category"])
axis.set(xlabel="Publications with cue (%)", title="Keyword profile of the full publication corpus")
axis.grid(axis="x", alpha=0.25)
save_figure(figure, figure_dir / "keyword_category_profile.png")
category_summary
